# Notebook 1: Data preprocessing and cleanup

In this notebook we will read relavent datasets from the data sources below and do some cleanup like handling misisng data, outliers, and scaling

| Source | Purpose | Granularity | Link |
|----------|----------|----------|------|
| Fjelstul World Cup Database | Historical World Cup-specific data | Tournament, match, squad, player | https://github.com/jfjelstul/worldcup/blob/master/data-csv |
| International Football Results Dataset | All international matches | Match-level | https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017 | 
| World Football Elo Ratings | Team strength ratings | Team-date level | https://www.kaggle.com/datasets/saifalnimri/international-football-elo-ratings |

We will use the following datasets from the above datasources
| Dataset | Relevance to 2026 World Cup Prediction |
|----------|----------|
| **matches.csv** | Contains historical FIFA World Cup match results, including participating teams, scores, and outcomes. This serves as the primary source for understanding how teams have historically performed against one another on the World Cup stage and can be used to derive features such as World Cup win rate, goal differential, and tournament progression. |
| **teams.csv** | Provides a standardized list of teams and team identifiers used throughout the World Cup datasets. This dataset is useful for joining information across tables and resolving naming inconsistencies between the World Cup, international match results, and Elo rating datasets. |
| **tournaments.csv** | Contains metadata about each World Cup tournament, including year, host nation, and tournament structure. This information can be used to study historical tournament trends, host-country effects, and changes in competition format over time. |
| **tournament_standings.csv** | Records each team's final placement and tournament performance. This dataset can be used to create historical strength indicators such as prior World Cup appearances, average finishing position, semifinal appearances, and championship counts. |
| **shootouts.csv** | Contains information about matches decided by penalty shootouts. While penalty shootouts are relatively rare, they are important for modeling knockout-stage outcomes and can provide insights into a team's historical performance under high-pressure situations. |
| **goalscorers.csv** | Provides goal-level information including scorer, scoring minute, own goals, and penalties. This dataset enables the creation of advanced features such as scoring frequency, offensive efficiency, goal timing patterns, and dependence on penalty goals. |
| **results.csv** | Contains over a century of international football match results across friendlies, qualifiers, continental tournaments, and World Cups. This is one of the most important datasets because it provides a large sample of recent and historical matches that can be used to calculate team form, rolling performance metrics, head-to-head records, and goal statistics leading into the 2026 tournament. |
| **eloratings.csv** | Contains Elo ratings, a widely used measure of national team strength based on historical match performance. Elo ratings summarize a team's quality into a single predictive metric and are often among the strongest individual predictors of match outcomes. Features such as pre-match Elo, Elo difference, and Elo trend can significantly improve predictive performance. |

In [2]:
import pandas as pd

In [3]:
# Download relevant datasets from jelstul World Cup Database 
# https://github.com/jfjelstul/worldcup/blob/master/data-csv
# to our own branch

matches = pd.read_csv("./data/matches.csv")
teams = pd.read_csv("./data/teams.csv")
tournaments = pd.read_csv("./data/tournaments.csv")
standings = pd.read_csv("./data/tournament_standings.csv")

for df_name, df in {
    "matches": matches,
    "teams": teams,
    "tournaments": tournaments,
    "standings": standings
}.items():
    print(f"\n{df_name}")
    print(df.shape)
    display(df.head())


matches
(1248, 37)


,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,...,away_team_score_margin,extra_time,penalty_shootout,score_penalties,home_team_score_penalties,away_team_score_penalties,result,home_team_win,away_team_win,draw
0,1,WC-1930,1930 FIFA Men's World Cup,M-1930-01,France vs Mexico,group stage,Group 1,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
1,2,WC-1930,1930 FIFA Men's World Cup,M-1930-02,United States vs Belgium,group stage,Group 4,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
2,3,WC-1930,1930 FIFA Men's World Cup,M-1930-03,Yugoslavia vs Brazil,group stage,Group 2,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0
3,4,WC-1930,1930 FIFA Men's World Cup,M-1930-04,Romania vs Peru,group stage,Group 3,1,0,0,...,-2,0,0,0-0,0,0,home team win,1,0,0
4,5,WC-1930,1930 FIFA Men's World Cup,M-1930-05,Argentina vs France,group stage,Group 1,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0



teams
(88, 14)


,key_id,team_id,team_name,team_code,mens_team,womens_team,federation_name,region_name,confederation_id,confederation_name,confederation_code,mens_team_wikipedia_link,womens_team_wikipedia_link,federation_wikipedia_link
0,1,T-01,Algeria,DZA,1,0,Algerian Football Federation,Africa,CF-2,Confederation of African Football,CAF,https://en.wikipedia.org/wiki/Algeria_national...,not applicable,https://en.wikipedia.org/wiki/Algerian_Footbal...
1,2,T-02,Angola,AGO,1,0,Angolan Football Federation,Africa,CF-2,Confederation of African Football,CAF,https://en.wikipedia.org/wiki/Angola_national_...,not applicable,https://en.wikipedia.org/wiki/Angolan_Football...
2,3,T-03,Argentina,ARG,1,1,Argentine Football Association,South America,CF-4,South American Football Confederation,CONMEBOL,https://en.wikipedia.org/wiki/Argentina_nation...,https://en.wikipedia.org/wiki/Argentina_women'...,https://en.wikipedia.org/wiki/Argentine_Footba...
3,4,T-04,Australia,AUS,1,1,Football Australia,Oceania,CF-1,Asian Football Confederation,AFC,https://en.wikipedia.org/wiki/Australia_men%27...,https://en.wikipedia.org/wiki/Australia_women'...,https://en.wikipedia.org/wiki/Football_Australia
4,5,T-05,Austria,AUT,1,0,Austrian Football Association,Europe,CF-6,Union of European Football Associations,UEFA,https://en.wikipedia.org/wiki/Austria_national...,not applicable,https://en.wikipedia.org/wiki/Austrian_Footbal...



tournaments
(30, 18)


,key_id,tournament_id,tournament_name,year,start_date,end_date,host_country,winner,host_won,count_teams,group_stage,second_group_stage,final_round,round_of_16,quarter_finals,semi_finals,third_place_match,final
0,1,WC-1930,1930 FIFA Men's World Cup,1930,1930-07-13,1930-07-30,Uruguay,Uruguay,1,13,1,0,0,0,0,1,0,1
1,2,WC-1934,1934 FIFA Men's World Cup,1934,1934-05-27,1934-06-10,Italy,Italy,1,16,0,0,0,1,1,1,1,1
2,3,WC-1938,1938 FIFA Men's World Cup,1938,1938-06-04,1938-06-19,France,Italy,0,15,0,0,0,1,1,1,1,1
3,4,WC-1950,1950 FIFA Men's World Cup,1950,1950-06-24,1950-07-16,Brazil,Uruguay,0,13,1,0,1,0,0,0,0,0
4,5,WC-1954,1954 FIFA Men's World Cup,1954,1954-06-16,1954-07-04,Switzerland,West Germany,0,16,1,0,0,0,1,1,1,1



standings
(120, 7)


,key_id,tournament_id,tournament_name,position,team_id,team_name,team_code
0,1,WC-1930,1930 FIFA Men's World Cup,1,T-84,Uruguay,URY
1,2,WC-1930,1930 FIFA Men's World Cup,2,T-03,Argentina,ARG
2,3,WC-1930,1930 FIFA Men's World Cup,3,T-83,United States,USA
3,4,WC-1930,1930 FIFA Men's World Cup,4,T-87,Yugoslavia,YUG
4,5,WC-1934,1934 FIFA Men's World Cup,1,T-41,Italy,ITA


In [4]:
# Check for missing values

print(matches.isnull().sum().sort_values(ascending=False))
print(teams.isnull().sum().sort_values(ascending=False))
print(tournaments.isnull().sum().sort_values(ascending=False))
print(standings.isnull().sum().sort_values(ascending=False))


key_id                       0
home_team_code               0
away_team_name               0
away_team_code               0
score                        0
home_team_score              0
away_team_score              0
home_team_score_margin       0
away_team_score_margin       0
extra_time                   0
penalty_shootout             0
score_penalties              0
home_team_score_penalties    0
away_team_score_penalties    0
result                       0
home_team_win                0
away_team_win                0
away_team_id                 0
home_team_name               0
tournament_id                0
home_team_id                 0
tournament_name              0
match_id                     0
match_name                   0
stage_name                   0
group_name                   0
group_stage                  0
knockout_stage               0
replayed                     0
replay                       0
match_date                   0
match_time                   0
stadium_

In [5]:
# Download International Football Results Dataset from Kaggle 
# https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017results = pd.read_csv("data/results.csv")

shootouts = pd.read_csv("./data/shootouts.csv")
goalscorers = pd.read_csv("./data/goalscorers.csv")
results = pd.read_csv("./data/results.csv")

for df_name, df in {
    "shootouts": shootouts,
    "goalscorers": goalscorers,
    "results": results
}.items():
    print(f"\n{df_name}")
    print(df.shape)
    display(df.head())


shootouts
(678, 5)


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN



goalscorers
(47606, 8)


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False



results
(49477, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
# Check for missing values

print(shootouts.isnull().sum().sort_values(ascending=False))
print(goalscorers.isnull().sum().sort_values(ascending=False))
print(results.isnull().sum().sort_values(ascending=False))

first_shooter    422
date               0
home_team          0
away_team          0
winner             0
dtype: int64
minute       256
scorer        48
date           0
home_team      0
away_team      0
team           0
own_goal       0
penalty        0
dtype: int64
home_score    70
away_score    70
date           0
home_team      0
away_team      0
tournament     0
city           0
country        0
neutral        0
dtype: int64


**first_shooter** in the 'shoutouts' dataset has 422 null values but we most likely will not use it in the model

**minute** missing minutes are common in older historical matches because the exact scoring minute wasn't recorded. Leave them as null for now

**scorer** should not be null, since there are only 48 out of 47606 rows (~.1%), it's ok to drop)

**home_score** and **away_score** both have 70 missing rows as shown in the next cell. They're all future upcoming matches so we'll drop them from the dataset used to train the model

In [7]:
# Check for the null home_score and away_score
results[
    results["home_score"].isna() &
    results["away_score"].isna()
]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49407,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False
49408,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False
49409,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
49410,2026-06-13,Brazil,Morocco,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49411,2026-06-13,Haiti,Scotland,NaN,NaN,FIFA World Cup,Foxborough,United States,True
...,...,...,...,...,...,...,...,...,...
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


In [8]:
# Drop future matches that don't already have scores
results = results.dropna(
    subset=["home_score", "away_score"]
)

In [9]:
# Check outliers from results
results["goal_diff"] = (
    results["home_score"] -
    results["away_score"]
).abs()

results["goal_diff"].describe()

count    49407.000000
mean         1.717469
std          1.793860
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         31.000000
Name: goal_diff, dtype: float64

In [10]:
display(results[
    results["goal_diff"] >= 20
])
print("Spot checked the really big goal diff matches and  they're all ligimimate matches just have extreme scores")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,goal_diff
6580,1966-04-03,Libya,Oman,21.0,0.0,Arab Cup,Baghdad,Iraq,True,21.0
8551,1971-09-13,Tahiti,Cook Islands,30.0,0.0,South Pacific Games,Papeete,Tahiti,False,30.0
11916,1979-08-30,Fiji,Kiribati,24.0,0.0,South Pacific Games,Nausori,Fiji,False,24.0
15922,1987-12-15,American Samoa,Papua New Guinea,0.0,20.0,South Pacific Games,Nouméa,New Caledonia,True,20.0
24181,2000-02-14,Kuwait,Bhutan,20.0,0.0,AFC Asian Cup qualification,Kuwait City,Kuwait,False,20.0
25422,2001-04-09,Australia,Tonga,22.0,0.0,FIFA World Cup qualification,Coffs Harbour,Australia,False,22.0
25425,2001-04-11,Australia,American Samoa,31.0,0.0,FIFA World Cup qualification,Coffs Harbour,Australia,False,31.0
27346,2003-06-30,Sark,Isle of Wight,0.0,20.0,Island Games,Saint Martin,Guernsey,True,20.0
29045,2005-03-11,Guam,North Korea,0.0,21.0,EAFF Championship,Taipei,Taiwan,True,21.0
30518,2006-11-24,Sápmi,Monaco,21.0,1.0,Viva World Cup,Hyères,France,True,20.0


Spot checked the really big goal diff matches and  they're all ligimimate matches just have extreme scores


In [11]:
# Download International Football Elo Ratings (1872-2025)
# https://www.kaggle.com/datasets/saifalnimri/international-football-elo-ratings

eloratings = pd.read_csv("./data/eloratings.csv")
print("eloratings shape:", eloratings.shape)
eloratings.head()

eloratings shape: (6678, 4)


,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8


In [12]:
print(eloratings.isnull().sum().sort_values(ascending=False))

rating    31
date       0
team       0
change     0
dtype: int64


In [13]:
display(eloratings[eloratings["rating"].isna()])
eloratings[eloratings["rating"].isna()].shape

eloratings[(eloratings["team"] == "Moldova") &
    (eloratings["rating"].notna())]

print("There are 32 rows for Moldova and 31 of them have NaN null rating, the other row has 0.0 for rating")


,date,team,rating,change
1219,10/14/1992,Moldova,NaN,-4
1346,9/6/1995,Moldova,NaN,-7
1536,10/7/1997,Moldova,NaN,-19
1775,9/1/2001,Moldova,NaN,15
1824,2/13/2002,Moldova,NaN,-32
1891,8/21/2002,Moldova,NaN,-6
2137,9/4/2004,Moldova,NaN,-10
2234,6/4/2005,Moldova,NaN,-10
2380,9/7/2005,Moldova,NaN,-15
2706,10/17/2007,Moldova,NaN,15


There are 32 rows for Moldova and 31 of them have NaN null rating, the other row has 0.0 for rating


### Clean up team names

In [14]:
# Find the unique team name from the Fjelstul World Cup Database and Elo Rating 

wc_teams = sorted(teams["team_name"].dropna().unique())
elo_teams = sorted(eloratings["team"].dropna().unique())

print("World Cup teams:", len(wc_teams))
print("Elo teams:", len(elo_teams))

World Cup teams: 88
Elo teams: 270


In [15]:
from difflib import get_close_matches

#Use fuzzy matching to suggest matches 
wc_not_in_elo = sorted(set(wc_teams) - set(elo_teams))
elo_not_in_wc = sorted(set(elo_teams) - set(wc_teams))

print(wc_not_in_elo[:50])
for team in wc_not_in_elo:
    print(team, "->", get_close_matches(team, elo_teams, n=3, cutoff=0.6))

['Bosnia and Herzegovina', 'Chinese Taipei', 'Costa Rica', 'Czech Republic', 'Dutch East Indies', 'East Germany', 'El Salvador', 'Equatorial Guinea', 'Ivory Coast', 'New Zealand', 'North Korea', 'Northern Ireland', 'Republic of Ireland', 'Saudi Arabia', 'Serbia and Montenegro', 'South Africa', 'South Korea', 'Soviet Union', 'Trinidad and Tobago', 'United Arab Emirates', 'United States', 'West Germany', 'Zaire']
Bosnia and Herzegovina -> ['Bosnia\xa0and\xa0Herzegovina']
Chinese Taipei -> []
Costa Rica -> ['Costa\xa0Rica']
Czech Republic -> ['Khmer\xa0Republic']
Dutch East Indies -> []
East Germany -> ['East\xa0Germany', 'West\xa0Germany', 'Germany']
El Salvador -> ['El\xa0Salvador']
Equatorial Guinea -> ['Equatorial\xa0Guinea']
Ivory Coast -> ['Ivory\xa0Coast']
New Zealand -> ['New\xa0Zealand', 'Netherlands', 'Swaziland']
North Korea -> ['North\xa0Korea', 'South\xa0Korea']
Northern Ireland -> ['Northern\xa0Ireland', 'Netherlands', 'Northern\xa0Mariana\xa0Islands']
Republic of Ireland ->

In [16]:
# Normalize teh elo names to get rid of the special character
eloratings["team"] = (
    eloratings["team"]
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)


wc_not_in_elo = sorted(
    set(teams["team_name"]) - set(eloratings["team"])
)

print(wc_not_in_elo)


['Chinese Taipei', 'Czech Republic', 'Dutch East Indies', 'Republic of Ireland', 'Zaire']


In [21]:
TEAM_NAME_MAPPING = {
    "Czech Republic": "Czechia",      # if present
    "Republic of Ireland": "Ireland", # if present
    "Chinese Taipei": "Taiwan"
}

teams["elo_team"] = teams["team_name"].replace(TEAM_NAME_MAPPING)
unmatched = sorted(set(teams["elo_team"]) - set(eloratings["team"]))
print("The two unmatched", unmatched, "do not have any close potential mapping suggestion")

The two unmatched ['Dutch East Indies', 'Zaire'] do not have any close potential mapping suggestion
